In [1]:
import sys
sys.path.append("..") 

In [2]:
import pandas as pd

In [3]:
# pip install kagglehub

In [4]:
from src.data_loader import load_and_prepare
from src.models import (
    MODEL_REGISTRY,
    train_model,
    evaluate_model,
    train_and_evaluate_all,
)

In [5]:
import src.models as m
print(m.__file__)
print(dir(m))

c:\Users\Twinkle\OneDrive\Desktop\ckd-prediction-system\notebooks\..\src\models.py
['DecisionTreeClassifier', 'KNeighborsClassifier', 'LGBMClassifier', 'LogisticRegression', 'MODEL_REGISTRY', 'RANDOM_STATE', 'RandomForestClassifier', 'SVC', 'XGBClassifier', '_HAS_LGBM', '_HAS_XGB', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_build_registry', 'accuracy_score', 'clone', 'confusion_matrix', 'evaluate_model', 'f1_score', 'pd', 'precision_score', 'recall_score', 'roc_auc_score', 'train_and_evaluate_all', 'train_model']


In [6]:
X_train, X_test, y_train, y_test, scaler = load_and_prepare(
    path="../data/processed/kidney_features.csv",
    strategy="smote",
)

In [7]:
print("Train:", X_train.shape, y_train.value_counts().to_dict())

Train: (400, 26) {1: 200, 0: 200}


In [8]:
print("Test :", X_test.shape,  y_test.value_counts().to_dict())

Test : (80, 26) {1: 50, 0: 30}


In [9]:
# !pip install xgboost lightgbm

In [10]:
list(MODEL_REGISTRY)

['logreg',
 'decision_tree',
 'random_forest',
 'knn',
 'svm',
 'xgboost',
 'lightgbm']

# Smoke-test one model end-to-end before running the full loop

In [11]:
logreg = train_model("logreg", X_train, y_train)
scores = evaluate_model(logreg, X_test, y_test)
scores

{'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'roc_auc': 1.0,
 'confusion_matrix': [[30, 0], [0, 50]]}

In [12]:
results_df, fitted_models = train_and_evaluate_all(X_train, y_train, X_test, y_test)
results_df.drop(columns="confusion_matrix")

c:\Users\Twinkle\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,model,accuracy,precision,recall,f1,roc_auc
0,logreg,1.0000,1.0,1.00,1.000000,1.000000
1,random_forest,1.0000,1.0,1.00,1.000000,1.000000
2,svm,1.0000,1.0,1.00,1.000000,1.000000
3,xgboost,0.9875,1.0,0.98,0.989899,1.000000
4,lightgbm,0.9875,1.0,0.98,0.989899,1.000000
5,decision_tree,0.9625,1.0,0.94,0.969072,0.970000
6,knn,0.9625,1.0,0.94,0.969072,0.999667


In [13]:
print("Train means:", X_train.mean().round(3).to_dict())
print("Train stds :", X_train.std().round(3).to_dict())
print("Test means :", X_test.mean().round(3).to_dict())
print("Test stds  :", X_test.std().round(3).to_dict())

Train means: {'age': -0.03, 'bp': -0.082, 'sg': 0.153, 'al': -0.136, 'su': -0.07, 'rbc': 0.91, 'pc': 0.842, 'pcc': 0.098, 'ba': 0.05, 'bgr': -0.093, 'bu': -0.101, 'sc': -0.075, 'sod': 0.08, 'pot': 0.003, 'hemo': 0.195, 'pcv': 0.16, 'wc': -0.036, 'rc': 0.154, 'htn': 0.292, 'dm': 0.255, 'cad': 0.062, 'appet': 0.845, 'pe': 0.145, 'ane': 0.122, 'bun_creatinine_ratio': 0.15, 'anemia_ckd_flag': 0.105}
Train stds : {'age': 0.984, 'bp': 0.943, 'sg': 0.963, 'al': 0.936, 'su': 0.906, 'rbc': 0.287, 'pc': 0.365, 'pcc': 0.297, 'ba': 0.218, 'bgr': 0.919, 'bu': 0.922, 'sc': 0.908, 'sod': 0.932, 'pot': 0.961, 'hemo': 0.995, 'pcv': 0.97, 'wc': 0.952, 'rc': 0.979, 'htn': 0.455, 'dm': 0.436, 'cad': 0.242, 'appet': 0.362, 'pe': 0.353, 'ane': 0.328, 'bun_creatinine_ratio': 1.102, 'anemia_ckd_flag': 0.307}
Test means : {'age': 0.112, 'bp': -0.108, 'sg': -0.006, 'al': 0.084, 'su': 0.296, 'rbc': 0.862, 'pc': 0.838, 'pcc': 0.038, 'ba': 0.025, 'bgr': 0.2, 'bu': 0.017, 'sc': -0.05, 'sod': 0.188, 'pot': 1.619, 'h

In [14]:
merged = X_train.merge(X_test, how="inner")
print("Exact duplicate rows between train/test:", len(merged))

Exact duplicate rows between train/test: 0


In [15]:
# Optional: cross-validate the top model to confirm scores hold up across folds,
# not just this one 80/20 split
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    MODEL_REGISTRY["logreg"], X_train, y_train, cv=cv, scoring="recall"
)
print(cv_scores, cv_scores.mean())

[1.    1.    0.975 1.    1.   ] 0.9949999999999999


In [16]:
results_df.to_csv("../data/processed/model_comparison.csv", index=False)
print("Saved comparison table to data/processed/model_comparison.csv")

Saved comparison table to data/processed/model_comparison.csv
